In [1]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm

In [2]:
test = torch.load("../Data/Modulated_Silicon_Data_400_structures.pt")

In [39]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, output_size, hidden_layers, hidden_nodes):
        super(MLP, self).__init__()
        layers = []
        layers.append(torch.nn.Linear(input_size, hidden_nodes))
        layers.append(torch.nn.ReLU())
        for _ in range(hidden_layers - 1):
            layers.append(torch.nn.Linear(hidden_nodes, hidden_nodes))
            layers.append(torch.nn.ReLU())
        layers.append(torch.nn.Linear(hidden_nodes, output_size))
        self.network = torch.nn.Sequential(*layers)

    def forward(self, x):
        if x.dim() == 3:
            amplitudes = x[:, :, 0]
            phases = x[:, :, 1]
            an = amplitudes * torch.cos(phases)
            bn = amplitudes * torch.sin(phases)
            x = torch.cat((an, bn),dim=1)
        return self.network(x)

In [52]:
#Make a simple dataloader to check the data
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(test['features'].float(), test['targets'].float())
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
for batch in dataloader:
    features, targets = batch
    print("Features shape:", features.shape)
    print("Targets shape:", targets.shape)
    break

Features shape: torch.Size([32, 5, 2])
Targets shape: torch.Size([32, 2])


In [53]:
x = batch[0]

amplitudes = x[:, :, 0]
phases = x[:, :, 1]
an = amplitudes * torch.cos(phases)
bn = amplitudes * torch.sin(phases)
x = torch.cat((an, bn),dim=1)

In [54]:
#train/test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [57]:
#train the MLP
input_size = 10  # 5 amplitudes + 5 phases
output_size = test['targets'].shape[1]  # Number of wavelengths
hidden_layers = 3
hidden_nodes = 128
model = MLP(input_size, output_size, hidden_layers, hidden_nodes)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 1000

# Use a simple range tqdm for epochs
pbar = tqdm(range(num_epochs), desc="Training")

for epoch in pbar:
    model.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Update the description with the average loss for this epoch
    avg_loss = running_loss / len(train_loader)
    pbar.set_description(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

In [58]:
inputs, targets = next(iter(test_loader))

In [59]:
inputs

tensor([[[23.2778,  3.5019],
         [ 1.6780,  2.5007],
         [19.3026,  3.9964],
         [35.2526,  4.0937],
         [14.9896,  4.8167]],

        [[ 2.6439,  2.4548],
         [14.7295,  3.7594],
         [30.0070,  5.6895],
         [ 8.3489,  3.0684],
         [15.2161,  5.6519]],

        [[30.6207,  1.3674],
         [38.8749,  0.4153],
         [ 6.9792,  2.3382],
         [25.5002,  3.0131],
         [28.1951,  4.3678]],

        [[19.8428,  2.6447],
         [ 9.5340,  5.8634],
         [17.6641,  2.5451],
         [30.8894,  0.4651],
         [27.5505,  5.8525]],

        [[ 4.2575,  1.7525],
         [ 3.3502,  2.3553],
         [20.6319,  0.0478],
         [24.6566,  6.2567],
         [ 0.3305,  2.8336]],

        [[25.1745,  3.8808],
         [36.0544,  6.2013],
         [ 5.7620,  5.5104],
         [38.1761,  5.9596],
         [29.4990,  0.9662]],

        [[26.1333,  3.4379],
         [10.4402,  1.2898],
         [32.1495,  2.0864],
         [28.1635,  4.0981],
  